# Task 14 — symWeighted gate, two independent Colab clones

This fail-closed runner verifies one immutable remote SHA, creates two fresh independent clone/output roots, and runs the preregistered judge in normal and optimized Python in each clone. It licenses only the exact entrywise `symWeighted` factorisation gate, not Lean or either analytic sector bound.

In [ ]:
import datetime, hashlib, json, os, platform, shlex, subprocess, tempfile, time, urllib.request
from pathlib import Path

REPO_URL = 'https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git'
TARGET_SHA = '2c009f607a6e0747f69effc0378b577b0f853ffb'
JUDGE_REL = 'scripts/judge_spatial_symweighted_factorization.py'
HARNESS_REL = 'scripts/run_spatial_symweighted_gate_detached.ps1'
JUDGE_SHA256 = 'a95e66da0ee527b1776ceb3d13d83760d1fd88cc9227ebea668a2b98ca1946cf'
HARNESS_SHA256 = '668681a7f29e2a228b8036be31472a0203d56edfd7f9de9a8753dc0449bc5a65'
EXPECTED_COUNT = 5460

def fail(message, payload=None):
    print(f'RESULT: FAIL: {message}')
    if payload is not None:
        print(payload)
    raise RuntimeError(message)

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    return sha256_bytes(path.read_bytes())

def run_checked(command, cwd=None):
    result = subprocess.run(command, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False)
    print('$', shlex.join(map(str, command)))
    print(result.stdout, end='')
    print(f'[exit {result.returncode}]')
    if result.returncode != 0:
        fail('setup command failed', {'command': command, 'exit': result.returncode})
    return result.stdout.strip()

# Verify the exact remote object and both versioned source hashes before cloning.
remote_hashes = {}
for rel, expected in ((JUDGE_REL, JUDGE_SHA256), (HARNESS_REL, HARNESS_SHA256)):
    url = f'https://raw.githubusercontent.com/lluiseriksson/THE-ERIKSSON-PROGRAMME/{TARGET_SHA}/{rel}'
    data = urllib.request.urlopen(url, timeout=60).read()
    actual = sha256_bytes(data)
    print(f'remote {TARGET_SHA} {rel} sha256={actual}')
    if actual != expected:
        fail('remote source hash mismatch', {'path': rel, 'actual': actual, 'expected': expected})
    remote_hashes[rel] = actual

record = {
    'classification': 'exact symWeighted gate, two independent Colab clones',
    'target_sha': TARGET_SHA, 'remote_hashes': remote_hashes,
    'utc_started': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'runtime': platform.platform(), 'python': platform.python_version(),
    'cpu_count': os.cpu_count(), 'clones': [], 'comparisons': {},
}

for clone_label in ('clone-a', 'clone-b'):
    root = Path(tempfile.mkdtemp(prefix=f'spatial-symweighted-{clone_label}-'))
    repo = root / 'repo'
    outputs = root / 'outputs'
    outputs.mkdir()
    run_checked(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(repo)])
    run_checked(['git', '-C', str(repo), 'checkout', '--detach', TARGET_SHA])
    head = run_checked(['git', '-C', str(repo), 'rev-parse', 'HEAD'])
    if head != TARGET_SHA:
        fail('clone HEAD mismatch', {'clone': clone_label, 'head': head})
    local_judge_hash = sha256_file(repo / JUDGE_REL)
    local_harness_hash = sha256_file(repo / HARNESS_REL)
    if local_judge_hash != JUDGE_SHA256 or local_harness_hash != HARNESS_SHA256:
        fail('clone source hash mismatch', {'clone': clone_label})

    clone_record = {
        'label': clone_label, 'root': str(root), 'repo': str(repo), 'head': head,
        'judge_sha256': local_judge_hash, 'harness_sha256': local_harness_hash,
        'modes': [],
    }
    for mode in ('normal', 'optimized'):
        log_path = outputs / f'{mode}.log'
        stderr_path = outputs / f'{mode}.stderr.log'
        exit_tmp = outputs / f'{mode}.exitcode.tmp'
        exit_path = outputs / f'{mode}.exitcode'
        pids_path = outputs / f'{mode}.pids.json'
        runner_path = outputs / f'{mode}.runner.sh'
        semantic_paths = (log_path, stderr_path, exit_tmp, exit_path, pids_path, runner_path)
        if any(path.exists() for path in semantic_paths):
            fail('stale mode output exists before launch', {'clone': clone_label, 'mode': mode})
        flags = '-O ' if mode == 'optimized' else ''
        runner = f'''#!/usr/bin/env bash
set -u
printf '{{"wrapper_pid":%s' "$$" > {shlex.quote(str(pids_path))}
python {flags}{shlex.quote(str(repo / JUDGE_REL))} > {shlex.quote(str(log_path))} 2> {shlex.quote(str(stderr_path))} &
child=$!
printf ',"child_pid":%s}}\n' "$child" >> {shlex.quote(str(pids_path))}
wait "$child"
code=$?
printf '%s\n' "$code" > {shlex.quote(str(exit_tmp))}
test -s {shlex.quote(str(exit_tmp))}
grep -Eq '^-?[0-9]+$' {shlex.quote(str(exit_tmp))}
mv {shlex.quote(str(exit_tmp))} {shlex.quote(str(exit_path))}
exit "$code"
'''
        runner_path.write_text(runner, encoding='utf-8', newline='\n')
        started_iso = datetime.datetime.now(datetime.timezone.utc).isoformat()
        t0 = time.perf_counter()
        process = subprocess.Popen(['bash', str(runner_path)], cwd=repo)
        wrapper_pid = process.pid
        shell_exit = process.wait()
        wall_seconds = time.perf_counter() - t0
        completed_iso = datetime.datetime.now(datetime.timezone.utc).isoformat()

        if not exit_path.exists() or exit_path.stat().st_size == 0:
            fail('missing or empty final exit code', {'clone': clone_label, 'mode': mode})
        exit_lines = exit_path.read_text(encoding='utf-8').splitlines()
        if len(exit_lines) != 1:
            fail('exit code is not exactly one line', {'clone': clone_label, 'mode': mode})
        try:
            persisted_exit = int(exit_lines[0])
        except ValueError:
            fail('exit code is not decimal', {'clone': clone_label, 'mode': mode})
        if persisted_exit != shell_exit or shell_exit != 0:
            fail('nonzero or discordant exit code', {'clone': clone_label, 'mode': mode,
                'shell_exit': shell_exit, 'persisted_exit': persisted_exit})

        log_text = log_path.read_text(encoding='utf-8')
        log_lines = log_text.splitlines()
        if len(log_lines) != 1:
            fail('log contains stale or extra output', {'clone': clone_label, 'mode': mode,
                'line_count': len(log_lines)})
        payload = json.loads(log_lines[0])
        expected_fields = (
            'configuration_pairs_checked', 'scale_mutations_rejected',
            'source_closing_bond_mutations_rejected',
            'target_closing_bond_mutations_rejected',
        )
        if payload.get('status') != 'PASS' or any(payload.get(k) != EXPECTED_COUNT for k in expected_fields):
            fail('PASS payload or counters invalid', {'clone': clone_label, 'mode': mode, 'payload': payload})
        if payload.get('ring_sizes') != [1, 2, 3, 4, 5, 6]:
            fail('ring-size set mismatch', {'clone': clone_label, 'mode': mode})
        if stderr_path.read_bytes() != b'':
            fail('stderr is nonempty', {'clone': clone_label, 'mode': mode})
        pids = json.loads(pids_path.read_text(encoding='utf-8'))
        if pids.get('wrapper_pid') != wrapper_pid or not isinstance(pids.get('child_pid'), int):
            fail('PID record mismatch', {'clone': clone_label, 'mode': mode, 'pids': pids})

        mode_record = {
            'mode': mode, 'started': started_iso, 'completed': completed_iso,
            'wall_seconds': wall_seconds, 'wrapper_pid': wrapper_pid,
            'child_pid': pids['child_pid'], 'shell_exit': shell_exit,
            'persisted_exit': persisted_exit, 'fresh_outputs_verified': True,
            'payload': payload, 'hashes': {
                f'{mode}.log': sha256_file(log_path),
                f'{mode}.stderr.log': sha256_file(stderr_path),
                f'{mode}.exitcode': sha256_file(exit_path),
            },
        }
        clone_record['modes'].append(mode_record)
        print(json.dumps({'clone': clone_label, **mode_record}, sort_keys=True))
    record['clones'].append(clone_record)

for mode in ('normal', 'optimized'):
    names = (f'{mode}.log', f'{mode}.stderr.log', f'{mode}.exitcode')
    for name in names:
        hashes = [next(m for m in clone['modes'] if m['mode'] == mode)['hashes'][name]
                  for clone in record['clones']]
        if len(set(hashes)) != 1:
            fail('cross-clone output hash mismatch', {'name': name, 'hashes': hashes})
        record['comparisons'][name] = {'match': True, 'sha256': hashes[0]}

for clone in record['clones']:
    normal = next(m for m in clone['modes'] if m['mode'] == 'normal')
    optimized = next(m for m in clone['modes'] if m['mode'] == 'optimized')
    if normal['hashes']['normal.log'] != optimized['hashes']['optimized.log']:
        fail('normal/optimized payload hash mismatch', {'clone': clone['label']})

record['utc_completed'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
record['status'] = 'PASS'
artifact = Path('/content/spatial_symweighted_two_clone_certificate.json')
artifact.write_text(json.dumps(record, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(f'certificate_sha256={sha256_file(artifact)}')
print('SPATIAL SYMWEIGHTED TWO-CLONE GATE PASS')
from google.colab import files
files.download(str(artifact))